In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from itertools import product
from datetime import datetime
import time

# Network

In [ ]:
# Load Network Files
network_name = "11-500"
network_path = f"data/network/{network_name}/"
node_df = pd.read_csv(network_path + "nodes.csv")
link_df = pd.read_csv(network_path + "edges.csv")

node_id_list = node_df['node_index'].tolist()
print(f"Number of nodes: {len(node_id_list)}")

left_node_id_list = [i for i in range(0, len(node_id_list)//2)]
right_node_id_list = [i for i in range(len(node_id_list)//2, len(node_id_list))]
print(f"Number of left nodes: {len(left_node_id_list)}")
print(f"Number of right nodes: {len(right_node_id_list)}")

left_mobility_node_id_list = [116, 117, 118, 119, 120]
right_mobility_node_id_list = [237, 238, 239, 240, 241]

In [ ]:
# Load Distance Matrix
distance_matrix = np.load(network_path + "dist_matrix.npy")
print(f"Distance matrix shape: {distance_matrix.shape}")

In [ ]:
# Walking Speed (m/s)
WALKING_SPEED = 1.33

# Street station transfer time (s)
STREET_STATION_TRANSFER_TIME = 60

# Create Walking Time Matrix (s)
walking_time_matrix = distance_matrix / WALKING_SPEED

# PT Router

In [ ]:
from src.pt.PTOperator import PTOperator

train_10_gtfs_dir = r"data/gtfs/train/train_headway_10/matched"
train_10_pt_control = PTOperator(train_10_gtfs_dir, print_logs=False)

train_20_gtfs_dir = r"data/gtfs/train/train_headway_20/matched"
train_20_pt_control = PTOperator(train_20_gtfs_dir, print_logs=False)

train_30_gtfs_dir = r"data/gtfs/train/train_headway_30/matched"
train_30_pt_control = PTOperator(train_30_gtfs_dir, print_logs=False)

# Variables

In [ ]:
DEFAULT_BOARDING_TIME = 30  # seconds

# Train Headway (min)
train_headway_list = [10, 20, 30]

# MoD Fleet Size
mod_fleet_size_list = [30, 50, 70, 90, 110, 130, 150]

# MaaS Platform Communication Strategy
maas_communication_strategy_list = ['default', 'TPCS']

# Random Seed
random_seed_list = [3, 6, 9]

# MoD Waiting Time Threshold (s)
mod_waiting_time_threshold_list = [300, 600, 900]
# MoD Detour Time Threshold (%)
mod_detour_time_threshold_list = [30, 60, 90]

# Demand Size
demand_size_list = [i for i in range(100, 1001, 100)]
# Demand Split Ratio (Intra Modal, %)
demand_split_ratio_list = [0, 20, 40, 60, 80]

# Total Simulation Time (s)
total_sim_time = [0, 10800]  # 3 hours
# Warm-up Time (s)
warmup_time = 3600  # 1 hour
# Simulation Time Period (s)
time_period = [warmup_time, total_sim_time[1]+warmup_time]  # 1h + 3h

amod_request_level_analysis_folder = "data/amod-sim-results"

demand_files_folder = "data/demand/11-500/amod"

amod_simulation_results_folder = "D:\\projects\\fleetpy\\github\\ptbroker\\studies\\j26-tpcs\\results"

In [ ]:
# All scenario combinations
all_scenario_combinations = list(product(
    random_seed_list,
    mod_fleet_size_list,
    demand_size_list,
    demand_split_ratio_list,
    maas_communication_strategy_list,
    mod_detour_time_threshold_list,
    mod_waiting_time_threshold_list,
    train_headway_list
))

In [ ]:
for scenario_combination in tqdm(all_scenario_combinations):
    (
        random_seed,
        fleet_size,
        demand_size,
        demand_split_ratio,
        broker_type,
        op_max_detour_time_factor,
        op_max_wait_time,
        train_headway
    ) = scenario_combination

    demand_filepath = os.path.join(demand_files_folder, f"amod_ds{demand_size}_dsr{demand_split_ratio}_rs{random_seed}.csv")

    scenario_name = f"amod-{demand_size}-{demand_split_ratio}-{fleet_size}-{broker_type}-{op_max_detour_time_factor}-{op_max_wait_time}-{train_headway}-{random_seed}-{time_period[0]}-{time_period[1]}"

    amod_request_level_analysis_results_filepath = os.path.join(amod_request_level_analysis_folder, scenario_name, 'amod_request_level_analysis_results.csv')

    amod_simulation_results_scenario_folder = os.path.join(amod_simulation_results_folder, scenario_name)

    # Load files
    demand = pd.read_csv(demand_filepath)
    amod_request_level_analysis_results = pd.read_csv(amod_request_level_analysis_results_filepath)
    user_stats = pd.read_csv(os.path.join(amod_simulation_results_scenario_folder, '1_user-stats.csv'))

    # Add new column
    amod_request_level_analysis_results['request_time'] = -1.0
    amod_request_level_analysis_results['fm_duration'] = -1.0
    amod_request_level_analysis_results['pt_duration'] = -1.0
    amod_request_level_analysis_results['lm_duration'] = -1.0

    amod_request_level_analysis_results['pt_start_time'] = -1.0
    amod_request_level_analysis_results['lm_start_time'] = -1.0

    # Sellect all inter-city requests
    intercity_request = amod_request_level_analysis_results[amod_request_level_analysis_results['rq_type'] == 'inter']

    for idx, request in intercity_request.iterrows():
        request_id = request['request_id']
        request_time = demand.loc[demand['request_id'] == request_id, 'rq_time'].values[0]
        
        # Process all served inter-city requests
        if request['served_by_amod'] == True:
            user_stat = user_stats[user_stats['request_id'] == request_id]
            fm_sub_user_stat = user_stat[user_stat['sub_trip_id'] == 5]
            pt_sub_user_stat = user_stat[user_stat['sub_trip_id'] == 6]
            lm_sub_user_stat = user_stat[user_stat['sub_trip_id'] == 7]

            pt_start_time = pt_sub_user_stat['earliest_pickup_time'].values[0]
            lm_start_time = lm_sub_user_stat['earliest_pickup_time'].values[0]
            lm_end_time = lm_sub_user_stat['dropoff_time'].values[0]

            fm_duration = pt_start_time - request_time
            pt_duration = lm_start_time - pt_start_time
            lm_duration = lm_end_time - lm_start_time + DEFAULT_BOARDING_TIME

        # ----------------------------------------
        # Process all unserved inter-city requests
        else:
            origin_node = demand.loc[demand['request_id'] == request_id, 'start'].values[0]
            destination_node = demand.loc[demand['request_id'] == request_id, 'end'].values[0]
            subnetwork = request['subnetwork']
   
            if subnetwork == 'left':
                start_station_street_node = 120
                start_station_id = "MH-L"
                end_station_street_node = 241
                end_station_id = "MH-R"
            else:
                start_station_street_node = 241
                start_station_id = "MH-R"
                end_station_street_node = 120
                end_station_id = "MH-L"
            fm_duration = walking_time_matrix[origin_node, start_station_street_node] + STREET_STATION_TRANSFER_TIME
            lm_duration = walking_time_matrix[end_station_street_node, destination_node] + STREET_STATION_TRANSFER_TIME

            pt_start_time = request_time + fm_duration
            # Conver to arrival datetime
            arrival_datetime = datetime(2024, 1, 1, 0, 0, 0) + pd.to_timedelta(pt_start_time, unit='s')

            if train_headway == 10:
                pt_control = train_10_pt_control
            elif train_headway == 20:
                pt_control = train_20_pt_control
            else:
                pt_control = train_30_pt_control

            pt_duration = pt_control.return_fastest_pt_journey_1to1(start_station_id, end_station_id, arrival_datetime, 3, detailed=False)['duration']
            lm_start_time = pt_start_time + pt_duration

        # Update the result dataframe
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'request_time'] = request_time
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'fm_duration'] = fm_duration
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'pt_duration'] = pt_duration
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'lm_duration'] = lm_duration
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'pt_start_time'] = pt_start_time
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'lm_start_time'] = lm_start_time

    # Select all intra-city requests
    intracity_request = amod_request_level_analysis_results[amod_request_level_analysis_results['rq_type'] == 'intra']

    for idx, request in intracity_request.iterrows():
        # Add request time
        request_id = request['request_id']
        request_time = demand.loc[demand['request_id'] == request_id, 'rq_time'].values[0]
        amod_request_level_analysis_results.loc[amod_request_level_analysis_results['request_id'] == request_id, 'request_time'] = request_time

    # Save the updated results
    amod_request_level_analysis_results.to_csv(amod_request_level_analysis_results_filepath, index=False)



  0%|          | 46/56700 [00:06<2:19:52,  6.75it/s]